# Income Classification Model

**Objective:** Build a classification model to identify individuals earning more than $50,000 using 40 demographic and employment-related variables.

This notebook handles data loading, preprocessing, and training an XGBoost classifier while accounting for the population weights provided in the dataset.

In [31]:
!/opt/anaconda3/bin/python -m pip install pandas numpy scikit-learn xgboost kmodes

In [32]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
import warnings
warnings.filterwarnings('ignore') # Keeps the output clean

## 1. Data Loading
The dataset is comma-delimited. We will first load the column names from `census-bureau.columns` and then apply them to `censusbureau.data`.

In [33]:
import os
print(os.listdir())

['.cursor', 'census-bureau.columns', 'classification.ipynb', 'ML-TakehomeProject.pdf', 'census-bureau.data', '.venv']


In [34]:
# Load column names
with open('census-bureau.columns', 'r') as f:
    columns = [line.strip() for line in f.readlines()]

# Load the main dataset
df = pd.read_csv('census-bureau.data', names=columns, header=None)

# Preview the first few rows
df.head()

,age,class of worker,detailed industry recode,detailed occupation recode,education,wage per hour,enroll in edu inst last wk,marital stat,major industry code,major occupation code,...,country of birth father,country of birth mother,country of birth self,citizenship,own business or self employed,fill inc questionnaire for veteran's admin,veterans benefits,weeks worked in year,year,label
0,73,Not in universe,0,0,High school graduate,0,Not in universe,Widowed,Not in universe or children,Not in universe,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,0,95,- 50000.
1,58,Self-employed-not incorporated,4,34,Some college but no degree,0,Not in universe,Divorced,Construction,Precision production craft & repair,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,2,52,94,- 50000.
2,18,Not in universe,0,0,10th grade,0,High school,Never married,Not in universe or children,Not in universe,...,Vietnam,Vietnam,Vietnam,Foreign born- Not a citizen of U S,0,Not in universe,2,0,95,- 50000.
3,9,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.
4,10,Not in universe,0,0,Children,0,Not in universe,Never married,Not in universe or children,Not in universe,...,United-States,United-States,United-States,Native- Born in the United States,0,Not in universe,0,0,94,- 50000.


## 2. Data Preprocessing
We separate our features, the target label (income > $50K), and the sample weight. [cite_start]The `weight` column represents how many people in the general population each record stands for[cite: 13]. We will split the data into training and testing sets, ensuring the target variable remains balanced.

In [35]:
# 1. Isolate features, weights, and target
X = df.drop(columns=['label', 'weight'])
weights = df['weight']

# 2. Convert label to binary: 1 for income > $50k, 0 for <= $50k.
y = np.where(df['label'].astype(str).str.contains('+', regex=False), 1, 0)

# 3. Identify column types for the pipeline
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
numerical_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# 4. FIRST SPLIT: Separate out 15% for the final TEST Set
X_temp, X_test, y_temp, y_test, w_temp, w_test = train_test_split(
    X, y, weights, test_size=0.15, random_state=42, stratify=y
)

# 5. SECOND SPLIT: Split the remaining 85% into TRAIN (70% total) and VAL (15% total)
# 0.15 / 0.85 is roughly 0.1764
X_train, X_val, y_train, y_val, w_train, w_val = train_test_split(
    X_temp, y_temp, w_temp, test_size=0.1764, random_state=42, stratify=y_temp
)

print(f"Training Data: {len(X_train)} rows")
print(f"Validation Data: {len(X_val)} rows")
print(f"Test Data: {len(X_test)} rows")

Training Data: 139677 rows
Validation Data: 29917 rows
Test Data: 29929 rows


## 3. Baseline Model Comparison (on Validation Set)
Before tuning a complex model, we establish a baseline. We train three models on the `Train` set and evaluate them on the `Validation` set.

In [39]:
import xgboost as xgb

# Create the preprocessing steps (used for all models)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
    ])

# Define the progression of models
models = {
    "1. Baseline: Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "2. Basic Non-Linear: Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "3. Ensemble (Voting): Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "4. Ensemble (Learning): XGBoost": xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

# Train and evaluate each model
for name, model in models.items():
    clf = Pipeline(steps=[('preprocessor', preprocessor), ('classifier', model)])
    
    # Train on TRAINING set
    clf.fit(X_train, y_train, classifier__sample_weight=w_train)
    
    # Evaluate on VALIDATION set
    y_pred_val = clf.predict(X_val)
    y_proba_val = clf.predict_proba(X_val)[:, 1]
    
    print(f"\n{'='*60}")
    print(f" {name}")
    print(f"{'='*60}")
    print(classification_report(y_val, y_pred_val, sample_weight=w_val))
    print(f"Validation ROC-AUC: {roc_auc_score(y_val, y_proba_val, sample_weight=w_val):.4f}")


 1. Baseline: Logistic Regression
              precision    recall  f1-score   support

           0       0.96      0.99      0.97 48839316.71000012
           1       0.72      0.38      0.50 3322447.3999999957

    accuracy                           0.95 52161764.11000012
   macro avg       0.84      0.69      0.74 52161764.11000012
weighted avg       0.94      0.95      0.94 52161764.11000012

Validation ROC-AUC: 0.9397

 2. Basic Non-Linear: Decision Tree
              precision    recall  f1-score   support

           0       0.95      0.99      0.97 48839316.71000012
           1       0.69      0.30      0.42 3322447.3999999957

    accuracy                           0.95 52161764.11000012
   macro avg       0.82      0.65      0.70 52161764.11000012
weighted avg       0.94      0.95      0.94 52161764.11000012

Validation ROC-AUC: 0.8946

 3. Ensemble (Voting): Random Forest
              precision    recall  f1-score   support

           0       0.96      0.99      0.97 4

## 4. Hyperparameter Tuning (Validation Set)
Assuming Random Forest proved to be the strongest candidate, we will now tune its hyperparameters to squeeze out maximum performance. We use a transparent loop to test combinations of `max_depth` and `n_estimators` on the Validation set.

In [40]:
# Define the settings to test for XGBoost
depths = [3, 5, 7]
learning_rates = [0.05, 0.1, 0.2]

best_score = 0
best_params = {}
best_xgb_model = None

print("--- Tuning XGBoost on Validation Set ---\n")

for d in depths:
    for lr in learning_rates:
        # Build pipeline
        clf = Pipeline(steps=[
            ('preprocessor', preprocessor),
            ('classifier', xgb.XGBClassifier(
                max_depth=d, 
                learning_rate=lr, 
                n_estimators=100, # Keeping the number of trees fixed for this test
                use_label_encoder=False, 
                eval_metric='logloss', 
                random_state=42
            ))
        ])
        
        # Train on TRAINING set
        clf.fit(X_train, y_train, classifier__sample_weight=w_train)
        
        # Predict on VALIDATION set
        y_proba_val = clf.predict_proba(X_val)[:, 1]
        score = roc_auc_score(y_val, y_proba_val, sample_weight=w_val)
        
        print(f"Tested max_depth={d}, learning_rate={lr} --> Validation ROC-AUC: {score:.4f}")
        
        # Save best model
        if score > best_score:
            best_score = score
            best_params = {'max_depth': d, 'learning_rate': lr}
            best_xgb_model = clf

print("\n=========================================")
print(f"WINNING XGBOOST SETTINGS: {best_params}")
print(f"BEST VALIDATION SCORE: {best_score:.4f}")
print("=========================================")

--- Tuning XGBoost on Validation Set ---

Tested max_depth=3, learning_rate=0.05 --> Validation ROC-AUC: 0.9421
Tested max_depth=3, learning_rate=0.1 --> Validation ROC-AUC: 0.9454
Tested max_depth=3, learning_rate=0.2 --> Validation ROC-AUC: 0.9474
Tested max_depth=5, learning_rate=0.05 --> Validation ROC-AUC: 0.9458
Tested max_depth=5, learning_rate=0.1 --> Validation ROC-AUC: 0.9482
Tested max_depth=5, learning_rate=0.2 --> Validation ROC-AUC: 0.9475
Tested max_depth=7, learning_rate=0.05 --> Validation ROC-AUC: 0.9481
Tested max_depth=7, learning_rate=0.1 --> Validation ROC-AUC: 0.9488
Tested max_depth=7, learning_rate=0.2 --> Validation ROC-AUC: 0.9470

WINNING XGBOOST SETTINGS: {'max_depth': 7, 'learning_rate': 0.1}
BEST VALIDATION SCORE: 0.9488


In [41]:
# Generate predictions on the unseen TEST set using our best_xgb_model
y_pred_test = best_xgb_model.predict(X_test)
y_proba_test = best_xgb_model.predict_proba(X_test)[:, 1]

print("--- Final Evaluation on Unseen Test Set ---")
print(classification_report(y_test, y_pred_test, sample_weight=w_test))

print("\n--- Final Test ROC-AUC Score ---")
print(f"{roc_auc_score(y_test, y_proba_test, sample_weight=w_test):.4f}")

--- Final Evaluation on Unseen Test Set ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.98 48682785.61000037
           1       0.76      0.46      0.57 3361938.1200000024

    accuracy                           0.96 52044723.73000038
   macro avg       0.86      0.72      0.77 52044723.73000038
weighted avg       0.95      0.96      0.95 52044723.73000038


--- Final Test ROC-AUC Score ---
0.9526


In [42]:
# 1. Combine Train and Validation sets
X_train_full = pd.concat([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])
w_train_full = pd.concat([w_train, w_val])

# 2. Retrain the winning pipeline on the fully combined dataset
print("Retraining best model on combined Train + Validation data...")
best_xgb_model.fit(X_train_full, y_train_full, classifier__sample_weight=w_train_full)

# 3. Generate predictions on the unseen TEST set
y_pred_test = best_xgb_model.predict(X_test)
y_proba_test = best_xgb_model.predict_proba(X_test)[:, 1]

# 4. Print Final Metrics
print("\n--- Final Evaluation on Unseen Test Set ---")
print(classification_report(y_test, y_pred_test, sample_weight=w_test))

print("\n--- Final Test ROC-AUC Score ---")
print(f"{roc_auc_score(y_test, y_proba_test, sample_weight=w_test):.4f}")

Retraining best model on combined Train + Validation data...

--- Final Evaluation on Unseen Test Set ---
              precision    recall  f1-score   support

           0       0.96      0.99      0.98 48682785.61000037
           1       0.74      0.46      0.57 3361938.1200000024

    accuracy                           0.95 52044723.73000038
   macro avg       0.85      0.73      0.77 52044723.73000038
weighted avg       0.95      0.95      0.95 52044723.73000038


--- Final Test ROC-AUC Score ---
0.9533
